# AI Content Detector — Binoculars σε δωρεάν GPU (Kaggle / Google Colab)

Τρέχει τη **μαζική ανάλυση** του εργαλείου μαζί με τη δεύτερη μέθοδο **Binoculars** σε κάρτα
γραφικών του cloud: λίγα δευτερόλεπτα ανά εργασία αντί για 1–2 λεπτά σε απλό υπολογιστή.
Βγάζει πίνακα (CSV) με σκορ ανά εργασία και τα ζεύγη εργασιών με κοινά αποσπάσματα.

> ⚠️ **Προσωπικά δεδομένα.** Οι εργασίες φοιτητών είναι προσωπικά δεδομένα (GDPR). Εδώ
> ανεβαίνουν σε υπολογιστές της Google / Kaggle. Πριν το χρησιμοποιήσεις, βεβαιώσου ότι το
> επιτρέπει ο κανονισμός του ιδρύματός σου — ή **αφαίρεσε ονόματα και ΑΜ** από τα αρχεία.
> Η τοπική εφαρμογή (`ai_detector_server.py`) δεν στέλνει τις εργασίες πουθενά.

**Βήματα**
1. Ενεργοποίησε GPU — *Kaggle:* Settings → Accelerator → **GPU T4 x2** (ή P100) και Internet **On**.
   *Colab:* Runtime → Change runtime type → **T4 GPU**.
2. Τρέξε τα κελιά με τη σειρά (*Run all*). Σε όσα ζητούν κάτι, θα δεις οδηγίες.
3. Κατέβασε τα αποτελέσματα στο τέλος. Ο δίσκος του notebook σβήνεται όταν κλείσει η συνεδρία.

Δωρεάν όρια (ενδεικτικά, αλλάζουν): Kaggle ~30 ώρες GPU/εβδομάδα · Colab ~12 ώρες ανά
συνεδρία, χωρίς εγγυημένη GPU σε ώρες αιχμής.

## 1. Έλεγχος κάρτας γραφικών

In [ ]:
import torch
if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Δεν βρέθηκε GPU — ενεργοποίησέ την (βλ. βήμα 1 παραπάνω) αλλιώς το Binoculars θα είναι αργό.")

## 2. Κώδικας του εργαλείου

Το repository είναι **δημόσιο**, οπότε το επόμενο κελί κάνει απλώς `git clone`.

Αν δεν υπάρχει πρόσβαση στο internet από το notebook: στο GitHub Code → Download ZIP.
*Colab:* θα σου ζητηθεί να το ανεβάσεις παρακάτω.
*Kaggle:* ανέβασέ το ως Dataset και πρόσθεσέ το στο notebook (Add Input).

In [ ]:
import glob, os, shutil, subprocess, zipfile

REPO = "ZafeiroBarda/academic-ai-detector"
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
IN_KAGGLE = os.path.exists("/kaggle")
# Kaggle / Colab — ή ο τρέχων φάκελος σε απλό Jupyter στον υπολογιστή σου
WORK = "/kaggle/working" if IN_KAGGLE else ("/content" if IN_COLAB else os.getcwd())
os.chdir(WORK)


def read_secret(name):
    try:
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def find_code_dir():
    hits = glob.glob(f"{WORK}/**/ai_detector_server.py", recursive=True)
    return os.path.dirname(hits[0]) if hits else None


code_dir = find_code_dir()
if not code_dir:
    url = f"https://github.com/{REPO}.git"
    r = subprocess.run(["git", "clone", "--depth", "1", url, "ai-detector"], capture_output=True, text=True)
    print("✅ git clone" if r.returncode == 0 else "⚠️ git clone απέτυχε — χρησιμοποίησε το ZIP")
    code_dir = find_code_dir()
if not code_dir:
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True) if IN_KAGGLE else []
    if IN_COLAB and not zips:
        from google.colab import files
        print("Ανέβασε το ZIP του repository (Code → Download ZIP):")
        zips = list(files.upload().keys())
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall(f"{WORK}/ai-detector-zip")
    code_dir = find_code_dir()
    if not code_dir and IN_KAGGLE:   # Kaggle αποσυμπιέζει μόνο του τα datasets
        hits = glob.glob("/kaggle/input/**/ai_detector_server.py", recursive=True)
        if hits:
            shutil.copytree(os.path.dirname(hits[0]), f"{WORK}/ai-detector", dirs_exist_ok=True)
            code_dir = f"{WORK}/ai-detector"

assert code_dir, "Δεν βρέθηκε ο κώδικας — δες το βήμα 2 (git clone ή ZIP)."
os.chdir(code_dir)
print("📁 Κώδικας:", code_dir)

## 3. Εγκατάσταση

Το `torch` υπάρχει ήδη σε Colab/Kaggle· εγκαθίστανται μόνο όσα λείπουν.

In [ ]:
import sys
# Με την ίδια Python που τρέχει το notebook· σταματά καθαρά αν αποτύχει
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-binoculars.txt"], check=True)
print("✅ Εγκατάσταση ολοκληρώθηκε")

## 4. Ρυθμίσεις & εργασίες

* `PRESET`: για **ελληνικές** εργασίες προτείνεται `krikri-8b` (ελληνικό μοντέλο του ΙΕΛ, 4-bit
  ~12 GB — χωράει σε T4 16 GB) ή `meltemi-7b`. Πολυγλωσσικά: `qwen2.5-1.5b` (~3 GB μνήμη GPU),
  `qwen2.5-0.5b` (ταχύτερο) ή `falcon-7b` (τα μοντέλα του paper, ~14 GB — οριακά).
* Η πρώτη λήψη των ελληνικών μοντέλων είναι ~30 GB και παίρνει μερικά λεπτά.
* Οι εργασίες (.docx / .pdf / .txt) μπαίνουν στον φάκελο `theses/`.
  *Colab:* θα σου ζητηθεί να τις ανεβάσεις. *Kaggle:* πρόσθεσε ένα Dataset με τις εργασίες
  (Add Input) και γράψε τη διαδρομή του στο `KAGGLE_INPUT`.

In [ ]:
PRESET = "krikri-8b"   # ή "meltemi-7b", "qwen2.5-1.5b", "qwen2.5-0.5b", "falcon-7b"
KAGGLE_INPUT = "/kaggle/input/theses"   # μόνο για Kaggle

os.environ["AI_DETECTOR_BINOCULARS"] = PRESET
if PRESET in ("krikri-8b", "meltemi-7b"):
    # 4-bit φόρτωση (μόνο GPU NVIDIA) ώστε τα δύο μοντέλα να χωράνε σε 16 GB
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
os.makedirs("theses", exist_ok=True)
if IN_COLAB:
    from google.colab import files
    print("Ανέβασε τις εργασίες:")
    for name, data in files.upload().items():
        with open(os.path.join("theses", os.path.basename(name)), "wb") as fh:
            fh.write(data)
elif IN_KAGGLE and os.path.isdir(KAGGLE_INPUT):
    shutil.copytree(KAGGLE_INPUT, "theses", dirs_exist_ok=True)
print(len(os.listdir("theses")), "αρχεία στο theses/")

## 5. (Προαιρετικό) Βαθμονόμηση ορίου Binoculars

Τα όρια του paper ισχύουν μόνο για `falcon-7b`. Για τα Qwen, χωρίς βαθμονόμηση βλέπεις μόνο το
score (χωρίς ετυμηγορία AI/άνθρωπος). Βάλε **γνωστά** κείμενα — ιδανικά 20+ ανά κατηγορία, στο
είδος των εργασιών σου — στους φακέλους `calibration/human/` και `calibration/ai/` και τρέξε το
κελί. Το όριο αποθηκεύεται στο `binoculars_config.json`· **κατέβασέ το** για να το
χρησιμοποιείς και τοπικά.

In [ ]:
if os.path.isdir("calibration/human") and os.path.isdir("calibration/ai"):
    subprocess.run([sys.executable, "tools/calibrate_binoculars.py",
                    "--human-dir", "calibration/human", "--ai-dir", "calibration/ai"], check=False)
else:
    print("Παραλείπεται: δεν υπάρχουν οι φάκελοι calibration/human και calibration/ai.")

## 6. Ανάλυση

In [ ]:
subprocess.run([sys.executable, "tools/batch_analyze.py", "theses", "--out", "results/results.csv"], check=True)

## 7. Αποτελέσματα

In [ ]:
import pandas as pd
df = pd.read_csv("results/results.csv")
display(df)
pairs = pd.read_csv("results/results_pairs.csv")
print("Ζεύγη με κοινά αποσπάσματα:", len(pairs))
display(pairs)

In [ ]:
shutil.make_archive(f"{WORK}/ai-detector-results", "zip", "results")
for extra in ("binoculars_config.json",):
    if os.path.exists(extra):
        shutil.copy(extra, f"{WORK}/{extra}")
if IN_COLAB:
    from google.colab import files
    files.download(f"{WORK}/ai-detector-results.zip")
else:
    print("Κατέβασε το ai-detector-results.zip από το Output (δεξιά) του Kaggle.")

## 8. Καθάρισμα

Σβήνει τις εργασίες από τον δίσκο του notebook (καλό να γίνεται πριν κλείσεις).

In [ ]:
shutil.rmtree("theses", ignore_errors=True)
shutil.rmtree("calibration", ignore_errors=True)
print("🧹 Οι εργασίες διαγράφηκαν από το notebook.")